In [1]:
import h5py
import numpy as np

# Percorso del file generato da EasySpin
file_path = "../data/50k/odmr_easyspin_raw_50000_DOPPIA_DENSITÀ.mat"

# Lettura tramite h5py (standard per MAT-file v7.3)
with h5py.File(file_path, "r") as f:
    # MATLAB salva i dataset trasposti (colonna-prioritaria vs riga-prioritaria)
    # Applichiamo .T per ripristinare la forma corretta: [N_spettri, N_punti]
    spectra_data = np.array(f["spectra_data"]).T
    labels_B = np.array(f["labels_B"]).squeeze()
    labels_D = np.array(f["labels_D"]).squeeze()
    labels_lw = np.array(f["labels_lw"]).squeeze()
    labels_c = np.array(f["labels_c"]).squeeze()
    labels_s = np.array(f["labels_s"]).squeeze()
    mw_freqs = np.array(f["mw_freqs"]).squeeze()

In [5]:
from ODMR_CNN.cnn_model import CNNPredictor

cnn = CNNPredictor("ODMR_CNN/artifacts/cnn_50k_48M")

# Conteggi osservati di fluorescenza: dip verso il basso.
# frequenze_MHz deve contenere la griglia 2750-2950 MHz, 335 punti.
pred = cnn.predict_counts(spectra_data, mw_freqs*1000)

c = pred["c"]                   # forma (N,), adimensionale
s = pred["s"]                   # forma (N,), separazione completa / 200 MHz
nu1 = pred["nu1_MHz"]           # forma (N,), frequenza inferiore prevista
nu2 = pred["nu2_MHz"]           # forma (N,), frequenza superiore prevista
centro = pred["center_MHz"]
splitting = pred["delta_MHz"]

# Da qui il collega utilizza c,s oppure nu1,nu2 nel PROPRIO codice.